# Instalaciones

In [1]:
# pdfplumber: leer el PDF de recetas MINSA
# openpyxl: leer los 3 Excel

#%pip install -q openpyxl pdfplumber

# Rutas de archivos


In [2]:
import os
CARPETA         = r"C:\Users\milu_\ricardo palma ia\Proyecto NutriAgente\Fuentes"
CARPETA_OUTPUT  = r"C:\Users\milu_\ricardo palma ia\Proyecto NutriAgente\Base_Conocimiento"
EXCEL_PLATOS    = os.path.join(CARPETA, "PLATOS PLANTA.xlsx")
EXCEL_SEMAFORO  = os.path.join(CARPETA, "SEMAFORO DE LA NUTRICION.xlsx")
EXCEL_PLANTILLA = os.path.join(CARPETA, "PLANTILLA SEMAFORO DE ALIMENTOS.xlsx")
PDF_MINSA       = os.path.join(CARPETA, "Guías_alimentarias para la poblacion peruana.pdf")
PDF_MINSA2      = os.path.join(CARPETA, "CENAN-0013.pdf")
SALIDA_PLATOS   = os.path.join(CARPETA_OUTPUT, "platos_kb.json")
SALIDA_REGLAS   = os.path.join(CARPETA_OUTPUT, "reglas_semaforo.json")

print("Verificando archivos:")
archivos = {
    "Excel Platos":    EXCEL_PLATOS,
    "Excel Semáforo":  EXCEL_SEMAFORO,
    "Excel Plantilla": EXCEL_PLANTILLA,
    "PDF MINSA":       PDF_MINSA,
    "PDF CENAN-0013":  PDF_MINSA2,
}
for nombre, ruta in archivos.items():
    existe = "✓" if os.path.exists(ruta) else "✗  ← REVISAR RUTA"
    print(f"  {existe} {nombre}: {ruta}")

Verificando archivos:
  ✓ Excel Platos: C:\Users\milu_\ricardo palma ia\Proyecto NutriAgente\Fuentes\PLATOS PLANTA.xlsx
  ✓ Excel Semáforo: C:\Users\milu_\ricardo palma ia\Proyecto NutriAgente\Fuentes\SEMAFORO DE LA NUTRICION.xlsx
  ✓ Excel Plantilla: C:\Users\milu_\ricardo palma ia\Proyecto NutriAgente\Fuentes\PLANTILLA SEMAFORO DE ALIMENTOS.xlsx
  ✓ PDF MINSA: C:\Users\milu_\ricardo palma ia\Proyecto NutriAgente\Fuentes\Guías_alimentarias para la poblacion peruana.pdf
  ✓ PDF CENAN-0013: C:\Users\milu_\ricardo palma ia\Proyecto NutriAgente\Fuentes\CENAN-0013.pdf


# Importaciones

In [3]:
import json
import re
import openpyxl
import pdfplumber
from typing import List, Dict, Optional

print("Librerías importadas ✓")

Librerías importadas ✓


# Normalizar Texto

In [4]:
def norm(texto: str) -> str:
    """Title Case respetando partículas españolas. Aplicar a TODOS los nombres."""
    MINUS = {"de","del","la","el","los","las","con","al","a","en","y","o","c/","s/","por"}
    palabras = str(texto).strip().split()
    return " ".join(
        p.capitalize() if i == 0 or p.lower() not in MINUS else p.lower()
        for i, p in enumerate(palabras)
    )

# Leer reglas del semáforo desde Excel

In [5]:
def leer_reglas_semaforo(path: str) -> dict:
    wb = openpyxl.load_workbook(path, data_only=True)
    ws = wb.active
    reglas = {}
    dim_actual = None

    for row in ws.iter_rows(values_only=True):
        col2, col3 = row[1], row[2]
        if not col2:
            continue
        if col3 is None:
            dim_actual = col2.strip()
            reglas[dim_actual] = {}
        elif dim_actual:
            color_raw = col2.strip().lower()
            if "verde" in color_raw:
                reglas[dim_actual]["verde"] = col3.strip()
            elif "ambar" in color_raw or "ámbar" in color_raw:
                reglas[dim_actual]["ambar"] = col3.strip()
            elif "rojo" in color_raw:
                reglas[dim_actual]["rojo"] = col3.strip()

    return reglas


reglas = leer_reglas_semaforo(EXCEL_SEMAFORO)

print(f"Dimensiones leídas: {list(reglas.keys())}")
print()
for dim, criterios in reglas.items():
    print(f"── {dim} ──")
    for color, desc in criterios.items():
        print(f"  [{color}] {desc}")
    print()

Dimensiones leídas: ['Energía', 'Carbohidratos', 'Proteína', 'Grasas', 'Fibra', 'Sodio']

── Energía ──
  [verde] Si la preparación contiene al menos 50% de verduras
  [ambar] Si la preparación contiene un tuberculo, guiso, o menestra
  [rojo] Si la preparación contiene fritura, apanado, plato a base de arroz o plato a base de fideos

── Carbohidratos ──
  [verde] Si la preparación contiene solo 1 carbohidrato en porción pequeña
  [ambar] Si la preparación contiene 2 carbohidratos cocidos o guisados.
  [rojo] Si la preparación está hecha base de arroz o plato a base de fideos

── Proteína ──
  [verde] Si la preparacion contiene un filete, preza, trozo de alimento de origen animal de aproximadamente 100g
  [ambar] Si la preparación contiene un alimentos de origen animal en picadilllo
  [rojo] Si no contiene alimento de origen animal

── Grasas ──
  [verde] Si la preparación no contiene frituras
  [ambar] Si la prepración es un salteado
  [rojo] Si la preparación contiene fritura profund

# Explorar hojas de PLATOS PLANTA.xlsx

In [6]:
wb_platos = openpyxl.load_workbook(EXCEL_PLATOS, data_only=True)

print(f"Hojas encontradas ({len(wb_platos.sheetnames)}):")
for nombre_hoja in wb_platos.sheetnames:
    ws = wb_platos[nombre_hoja]
    print(f"  - {nombre_hoja}: {ws.max_row} filas x {ws.max_column} columnas")

Hojas encontradas (10):
  - Complementos: 34 filas x 30 columnas
  - Sopas y entradas: 50 filas x 7 columnas
  - Pollo: 42 filas x 11 columnas
  - Res: 32 filas x 8 columnas
  - Pescado: 14 filas x 9 columnas
  - Cerdo y Pavita: 22 filas x 9 columnas
  - Víceras: 19 filas x 9 columnas
  - Arroces y fideos: 18 filas x 8 columnas
  - dieta: 24 filas x 6 columnas
  - AMANECIDA: 26 filas x 8 columnas


# Ver primeras filas de cada hoja

In [7]:
for nombre_hoja in wb_platos.sheetnames:
    ws = wb_platos[nombre_hoja]
    print(f"\n{'='*50}")
    print(f"HOJA: {nombre_hoja}")
    print('='*50)
    filas_mostradas = 0
    for row in ws.iter_rows(values_only=True):
        if any(c is not None for c in row):
            print(row)
            filas_mostradas += 1
        if filas_mostradas >= 4:
            break


HOJA: Complementos
(None, 'CEREALES', 'Avena', 'Quinua', '7 semillas', 'Kiwicha', 'Maca c/ leche', 'Soya', None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None)
(None, None, 'Chocolate', 'Manzana', 'Naranja', 'Piña', None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None)
(None, None, 'Manzana', 'Maracuyá', 'Durazno', 'Membrillo', None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None)
(None, None, 'Maracuyá', 'Naranja', 'Membrillo', 'Naranja', None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None)

HOJA: Sopas y entradas
(None, 'SOPAS / CALDOS', 'OBS.', None, None, 'ENTRADAS', 'OBS.')
(1, 'Caldo de pollo', 'No mezclar con fideos', None, 1, 'Wantan

# Extraer platos de PLATOS PLANTA.xlsx

In [8]:
# ── Mapeos para la nueva estructura ─────────────────────────────────
INSUMO_POR_HOJA = {
    "Pollo":            "Pollo",
    "Res":              "Res",
    "Pescado":          "Pescado",
    "Cerdo y Pavita":   "Cerdo y Pavita",
    "Víceras":          "Vísceras",
    "Arroces y fideos": "Arroz/Fideos",
}

def extraer_hoja_estandar(ws, nombre_hoja: str) -> List[Dict]:
    """Hojas con fila header que contiene 'PLATO' seguida de guarniciones/cárnicos."""
    platos = []
    plato_col = None
    extra_cols = []
    insumo = INSUMO_POR_HOJA.get(nombre_hoja, "Varios")

    for row in ws.iter_rows(values_only=True):
        row = list(row)

        # Detectar fila header
        if plato_col is None:
            for j, cell in enumerate(row):
                if cell and str(cell).strip().upper() == "PLATO":
                    plato_col = j
                    for k in range(j + 1, len(row)):
                        if row[k] and any(
                            kw in str(row[k]).upper()
                            for kw in ["GUARNICIÓN", "GUARNICION", "CARNICO", "CÁRNICO"]
                        ):
                            extra_cols.append(k)
                    break
            continue

        nombre_raw = row[plato_col] if plato_col < len(row) else None
        if not nombre_raw or str(nombre_raw).strip().upper() == "PLATO":
            continue
        nombre = str(nombre_raw).strip()
        extras = [
            str(row[k]).strip()
            for k in extra_cols
            if k < len(row) and row[k] and str(row[k]).strip()
        ]
        platos.append({
            "nombre": norm(nombre),      
            "insumo":      insumo,
            "rol":         "Fondo",
            "turno":       "Almuerzo",
            "guarniciones": extras,
            "fuente":      "PLATOS PLANTA.xlsx",
        })
    return platos


def extraer_sopas_entradas(ws) -> List[Dict]:
    """Hoja con dos columnas paralelas: SOPAS (col 1) y ENTRADAS (col 5)."""
    platos = []
    encabezado_visto = False
    for row in ws.iter_rows(values_only=True):
        row = list(row)
        if not encabezado_visto:
            if any(cell and "SOPA" in str(cell).upper() for cell in row):
                encabezado_visto = True
            continue
        if len(row) > 1 and row[1] and str(row[1]).strip():
            platos.append({
                "nombre": norm(str(row[1]).strip()),   
                "insumo":       "Varios",
                "rol":          "Sopa",
                "turno":        "Almuerzo",
                "guarniciones": [],
                "fuente":       "PLATOS PLANTA.xlsx",
            })
        if len(row) > 5 and row[5] and str(row[5]).strip():
            platos.append({
                "nombre": norm(str(row[5]).strip()),   
                "insumo":       "Varios",
                "rol":          "Entrada",
                "turno":        "Almuerzo",
                "guarniciones": [],
                "fuente":       "PLATOS PLANTA.xlsx",
            })
    return platos


def extraer_dieta(ws) -> List[Dict]:
    """Hoja dieta: platos con descripción completa, alternativas saludables."""
    platos = []
    for row in ws.iter_rows(values_only=True):
        row = list(row)
        for cell in row:
            if cell and isinstance(cell, str) and len(cell.strip()) > 5:
                texto = cell.strip()
                if texto in ("Pollo", "Res", "Pescado", "Cerdo y pavita", "Víceras"):
                    continue
                platos.append({
                    "nombre":       norm(texto),
                    "insumo":       "Varios",
                    "rol":          "Dieta",
                    "turno":        "Almuerzo",
                    "guarniciones": [],
                    "fuente":       "PLATOS PLANTA.xlsx",
                })
    return platos


def extraer_amanecida(ws) -> List[Dict]:
    """Hoja AMANECIDA: platos del turno Amanecida."""
    MESES = {
        "ENERO","FEBRERO","MARZO","ABRIL","MAYO","JUNIO",
        "JULIO","AGOSTO","SEPTIEMBRE","OCTUBRE","NOVIEMBRE","DICIEMBRE"
    }
    platos = []
    for row in ws.iter_rows(values_only=True):
        row = list(row)
        for cell in row:
            if cell and isinstance(cell, str) and len(cell.strip()) > 5:
                texto = cell.strip()
                if texto.upper() in MESES:         
                    continue
                platos.append({
                    "nombre":       norm(texto),
                    "insumo":       "Varios",
                    "rol":          "Fondo",
                    "turno":        "Amanecida",
                    "guarniciones": [],
                    "fuente":       "PLATOS PLANTA.xlsx",
                })
    return platos


# ── Ejecutar extracción ──────────────────────────────────────────────
todos_platos: List[Dict] = []

hojas_estandar = ["Pollo", "Res", "Pescado", "Cerdo y Pavita", "Víceras", "Arroces y fideos"]
for hoja in hojas_estandar:
    ws = wb_platos[hoja]
    extraidos = extraer_hoja_estandar(ws, hoja)
    todos_platos.extend(extraidos)
    print(f"  {hoja}: {len(extraidos)} platos")

ws = wb_platos["Sopas y entradas"]
extraidos = extraer_sopas_entradas(ws)
todos_platos.extend(extraidos)
print(f"  Sopas y entradas: {len(extraidos)} platos")

ws = wb_platos["dieta"]
extraidos = extraer_dieta(ws)
todos_platos.extend(extraidos)
print(f"  dieta: {len(extraidos)} platos")

ws = wb_platos["AMANECIDA"]
extraidos = extraer_amanecida(ws)
todos_platos.extend(extraidos)
print(f"  AMANECIDA: {len(extraidos)} platos")

print(f"\nTOTAL platos extraídos del Excel: {len(todos_platos)}")
print("\nEjemplo (primeros 3):")
for p in todos_platos[:3]:
    print(p)

  Pollo: 39 platos
  Res: 19 platos
  Pescado: 7 platos
  Cerdo y Pavita: 20 platos
  Víceras: 8 platos
  Arroces y fideos: 15 platos
  Sopas y entradas: 88 platos
  dieta: 46 platos
  AMANECIDA: 109 platos

TOTAL platos extraídos del Excel: 351

Ejemplo (primeros 3):
{'nombre': 'Ceviche de Pollo', 'insumo': 'Pollo', 'rol': 'Fondo', 'turno': 'Almuerzo', 'guarniciones': ['Frejol', 'Camote', 'Alverja', 'Yuca', 'Lenteja'], 'fuente': 'PLATOS PLANTA.xlsx'}
{'nombre': 'Escabeche de Pollo', 'insumo': 'Pollo', 'rol': 'Fondo', 'turno': 'Almuerzo', 'guarniciones': ['Frejol', 'Camote', 'Alverja', 'Yuca', 'Lenteja'], 'fuente': 'PLATOS PLANTA.xlsx'}
{'nombre': 'Adobo de Pollo', 'insumo': 'Pollo', 'rol': 'Fondo', 'turno': 'Almuerzo', 'guarniciones': ['Yuca', 'Camote'], 'fuente': 'PLATOS PLANTA.xlsx'}


# Extraer recetas del Apéndice 1 del PDF MINSA

In [9]:
# CELDA 8 – Extractor PDF con normalización de tildes

import unicodedata

def normalizar(texto: str) -> str:
    """Quita tildes y pasa a mayúsculas para comparar sin duplicar claves."""
    return ''.join(
        c for c in unicodedata.normalize('NFD', texto.upper())
        if unicodedata.category(c) != 'Mn'
    )

SECCIONES_CENAN = {
    "PESCADO":   "Pescado",
    "POTA":      "Pota",
    "MENESTRA":  "Menestras",
    "POLLO":     "Pollo",
    "SANGRE":    "Sangrecita",
    "HIGADO":    "Hígado",
    "VISCERA":   "Vísceras",
    "CARNE":     "Res",
}

RUIDO_NOMBRE = {
    "republica del peru", "ministerio", "instituto", "centro nacional",
    "peru de salud", "de alimentacion y nutricion", "raciones",
    "con ensalada", "con fruta", "refresco", "anexo", "indice",
    "introduccion", "mensaje", "nacional de salud",
    "universidad", "colegio de nutricion", "direccion",
    "bibliografia", "participaron", "elaboracion",
    "revision", "validacion", "red de salud", "red de lima",
    "de las comidas", "para el almuerzo", "nutritivas para"
}

def es_ruido(linea: str) -> bool:
    l = normalizar(linea).lower()   # ← FIX: comparar en minúsculas, no mayúsculas
    if not l or len(l) < 4:
        return True
    if linea.strip().startswith(("•", "-", "·")):
        return True
    if len(linea.strip()) > 70:
        return True
    if re.match(r"^\d", linea.strip()):
        return True
    return any(kw in l for kw in RUIDO_NOMBRE)

def limpiar_nombre(texto: str) -> str:
    # FIX PDF 2: "Perú de Salud... 4 raciones Nombre Del Plato" → extraer solo el nombre
    match = re.search(r'\d+\s+raciones?\s+(.+)', texto, re.IGNORECASE)
    if match:
        return match.group(1).strip().title()
    # PDF 1: "Texto previo. Nombre Del Plato" → tomar última parte
    if ". " in texto:
        partes = texto.split(". ")
        ultima = partes[-1].strip()
        if len(ultima) > 5:
            return ultima.title()
    return texto.strip().title()
    
def buscar_nombre_hacia_atras(lineas, desde, ventana=6):
    fragmentos = []
    for j in range(desde - 1, max(desde - ventana, -1), -1):
        candidato = lineas[j].strip()
        if es_ruido(candidato):
            if fragmentos:
                break
            continue
        fragmentos.insert(0, candidato)
        if len(fragmentos) >= 2:
            break
    nombre = limpiar_nombre(" ".join(fragmentos).strip())  # ← aplicar al final también
    return nombre if len(nombre) > 3 else None
    
def detectar_seccion_cenan(linea: str) -> Optional[str]:
    l = normalizar(linea)
    if "RECETAS A BASE DE" in l or "RECETAS DE" in l:
        for clave, categoria in SECCIONES_CENAN.items():
            if clave in l:
                return categoria
    return None

def nombre_valido(nombre: str) -> bool:
    """Descarta nombres que claramente son texto de ingredientes o autores."""
    sospechosos = ["yodada", "pimienta", "orégano", "oregano", "comino",
                   "magipo", "chumpitaz", "novoa", "saavedra"]
    return not any(s in nombre.lower() for s in sospechosos)
    
def extraer_recetas_pdf(path: str, fuente: str) -> List[Dict]:
    recetas = []
    seccion_actual = "Varios"
    with pdfplumber.open(path) as pdf:
        for pagina in pdf.pages:
            texto = pagina.extract_text() or ""
            lineas = texto.split("\n")
            for i, linea in enumerate(lineas):
                nueva_sec = detectar_seccion_cenan(linea)
                if nueva_sec:
                    seccion_actual = nueva_sec
                    continue
                es_patron1 = bool(re.search(r"[Ii]ngredientes para \d+", linea))
                es_patron2 = bool(re.search(r"[Ii]ngredientes\s+[Pp]reparaci", linea))
                if es_patron1 or es_patron2:
                    nombre = buscar_nombre_hacia_atras(lineas, i)
                    if nombre and nombre_valido(nombre):
                        recetas.append({
                            "nombre":       nombre,
                            "insumo":       seccion_actual,   # ← viene de SECCIONES_CENAN
                            "rol":          "Fondo",
                            "turno":        "Almuerzo",
                            "guarniciones": [],
                            "fuente":       fuente,
                        })
    vistos = set()
    unicos = []
    for r in recetas:
        if r["nombre"].lower() not in vistos:
            vistos.add(r["nombre"].lower())
            unicos.append(r)
    return unicos

recetas_pdf1 = extraer_recetas_pdf(PDF_MINSA,  "Guías_alimentarias MINSA 2023")
recetas_pdf2 = extraer_recetas_pdf(PDF_MINSA2, "CENAN-0013 MINSA 2023")

print(f"PDF 1 → {len(recetas_pdf1)} recetas:")
for r in recetas_pdf1:
    print(f"  [{r['insumo']}] {r['nombre']}")

print(f"\nPDF 2 → {len(recetas_pdf2)} recetas (primeras 15):")
for r in recetas_pdf2[:15]:
    print(f"  [{r['insumo']}] {r['nombre']}")
    

PDF 1 → 12 recetas:
  [Varios] Seco De Pollo Con Yuquitas
  [Varios] Higaditos De Pollo A La Jardinera Con Sarandaja
  [Varios] Sudado De Pescado Con Lentejitas
  [Varios] Chanfainita
  [Varios] Locro De Zapallo Con Huevo Frito
  [Varios] Carne De Res Con Rebosado De Trigo
  [Varios] Picante De Arvejas Con Tallarines Y Guiso De Pollo
  [Varios] Juane De Pollo
  [Varios] Caigua Rellena Con Carne Molida Y Tarwi
  [Varios] Escabeche De Pescado
  [Varios] Ajiaco De Olluco Con Carne
  [Varios] Picante Criollo De Pota

PDF 2 → 88 recetas (primeras 15):
  [Pescado] Adobo De Anchoveta
  [Pescado] Arvejitas Con Anchoveta Rebozada
  [Pescado] Arroz Chaufa Con Pescado
  [Pescado] Bolitas De Pescado
  [Pescado] Chicharrón De Pescado
  [Pescado] Cubierto De Pescado
  [Pescado] Escabeche De Pescado
  [Pescado] Frijoles Guisados Con Pescado
  [Pescado] Guiso De Trigo Con Pescado
  [Pescado] Hamburguesa De Pescado
  [Pescado] Pachamanca De Pescado A La Olla
  [Pescado] Pescado Al Ajo
  [Pescado] Reboz

# Extraer Complementos con tipos correctos y sin notas


In [10]:
TURNO_POR_CATEGORIA_COMPL = {
    "Cereales":       "Desayuno",
    "Postre 1":       "Almuerzo",
    "Refresco":       "Almuerzo",
    "Caldos Fuertes": "Amanecida",  
    "Ensalada":       "Almuerzo",
}

ROL_POR_CATEGORIA_COMPL = {
    "Cereales":       "Cereal",
    "Postre 1":       "Postre",
    "Refresco":       "Refresco",
    "Caldos Fuertes": "Sopa",
    "Ensalada":       "Ensalada",
}

ITEMS_EXCLUIR = {"no con menestra", "mixta"}

HEADERS_COMPL = {
    "CEREALES", "CALDOS FUERTES", "ENSALADA", 
    "POSTRE 1", "REFRESCO"
}

def extraer_complementos(ws) -> List[Dict]:
    items = []
    vistos = set()
    categoria_actual = "Complementos"

    for row in ws.iter_rows(values_only=True):
        for cell in row:
            if not cell or not isinstance(cell, str):
                continue
            texto = cell.strip()
            if len(texto) < 2:
                continue
            if texto.upper() in HEADERS_COMPL:        # ← reemplaza texto.isupper()
                categoria_actual = texto.title()
                continue
            if texto.lower() in ITEMS_EXCLUIR:
                continue
            if texto.lower() not in vistos:
                vistos.add(texto.lower())
                rol   = ROL_POR_CATEGORIA_COMPL.get(categoria_actual, "Complemento")
                turno = TURNO_POR_CATEGORIA_COMPL.get(categoria_actual, "Almuerzo")
                items.append({
                    "nombre":       norm(texto),
                    "insumo":       categoria_actual,
                    "rol":          rol,
                    "turno":        turno,
                    "guarniciones": [],
                    "fuente":       "PLATOS PLANTA.xlsx",
                })
    return items

complementos = extraer_complementos(wb_platos["Complementos"])
print(f"Complementos: {len(complementos)}")
for c in complementos[:5]:
    print(c)

Complementos: 93
{'nombre': 'Avena', 'insumo': 'Cereales', 'rol': 'Cereal', 'turno': 'Desayuno', 'guarniciones': [], 'fuente': 'PLATOS PLANTA.xlsx'}
{'nombre': 'Quinua', 'insumo': 'Cereales', 'rol': 'Cereal', 'turno': 'Desayuno', 'guarniciones': [], 'fuente': 'PLATOS PLANTA.xlsx'}
{'nombre': '7 Semillas', 'insumo': 'Cereales', 'rol': 'Cereal', 'turno': 'Desayuno', 'guarniciones': [], 'fuente': 'PLATOS PLANTA.xlsx'}
{'nombre': 'Kiwicha', 'insumo': 'Cereales', 'rol': 'Cereal', 'turno': 'Desayuno', 'guarniciones': [], 'fuente': 'PLATOS PLANTA.xlsx'}
{'nombre': 'Maca c/ Leche', 'insumo': 'Cereales', 'rol': 'Cereal', 'turno': 'Desayuno', 'guarniciones': [], 'fuente': 'PLATOS PLANTA.xlsx'}


# Unificar todo y guardar JSONs

In [11]:
todas_las_fuentes = todos_platos + recetas_pdf1 + recetas_pdf2 + complementos

vistos = set()
platos_kb = []
for p in todas_las_fuentes:
    clave = p["nombre"].lower().strip()
    if clave not in vistos:
        vistos.add(clave)
        platos_kb.append(p)

print(f"Total platos únicos: {len(platos_kb)}")
print(f"  Excel (platos):   {len(todos_platos)}")
print(f"  PDF MINSA 2023:   {len(recetas_pdf1)}")
print(f"  PDF CENAN-0013:   {len(recetas_pdf2)}")
print(f"  Complementos:     {len(complementos)}")
print(f"  Duplicados elim.: {len(todas_las_fuentes) - len(platos_kb)}")

# Verificar estructura del nuevo formato
print("\nEjemplos por turno:")
for turno in ["Almuerzo", "Amanecida", "Desayuno"]:
    ejs = [p for p in platos_kb if p["turno"] == turno][:2]
    for e in ejs:
        print(f"  [{turno}] {e['nombre']} | insumo={e['insumo']} | rol={e['rol']}")

with open(SALIDA_PLATOS, "w", encoding="utf-8") as f:
    json.dump(platos_kb, f, ensure_ascii=False, indent=2)
print(f"\n✓ {SALIDA_PLATOS}")

with open(SALIDA_REGLAS, "w", encoding="utf-8") as f:
    json.dump(reglas, f, ensure_ascii=False, indent=2)
print(f"✓ {SALIDA_REGLAS}")

Total platos únicos: 512
  Excel (platos):   351
  PDF MINSA 2023:   12
  PDF CENAN-0013:   88
  Complementos:     93
  Duplicados elim.: 32

Ejemplos por turno:
  [Almuerzo] Ceviche de Pollo | insumo=Pollo | rol=Fondo
  [Almuerzo] Escabeche de Pollo | insumo=Pollo | rol=Fondo
  [Amanecida] Escabeche de Pollo Jugoso c/ Camote | insumo=Varios | rol=Fondo
  [Amanecida] Caldo de Gallina Presa + Papa Amarilla + Cancha | insumo=Varios | rol=Fondo
  [Desayuno] Avena | insumo=Cereales | rol=Cereal
  [Desayuno] Quinua | insumo=Cereales | rol=Cereal

✓ C:\Users\milu_\ricardo palma ia\Proyecto NutriAgente\Base_Conocimiento\platos_kb.json
✓ C:\Users\milu_\ricardo palma ia\Proyecto NutriAgente\Base_Conocimiento\reglas_semaforo.json


In [12]:
from collections import Counter
turnos = Counter((p["turno"], p["rol"]) for p in platos_kb)
for (turno, rol), count in sorted(turnos.items()):
    print(f"  turno={turno:15} rol={rol:15} → {count} platos")



  turno=Almuerzo        rol=Dieta           → 46 platos
  turno=Almuerzo        rol=Ensalada        → 28 platos
  turno=Almuerzo        rol=Entrada         → 47 platos
  turno=Almuerzo        rol=Fondo           → 198 platos
  turno=Almuerzo        rol=Postre          → 39 platos
  turno=Almuerzo        rol=Refresco        → 5 platos
  turno=Almuerzo        rol=Sopa            → 41 platos
  turno=Amanecida       rol=Fondo           → 92 platos
  turno=Amanecida       rol=Sopa            → 2 platos
  turno=Desayuno        rol=Cereal          → 14 platos
